In [ ]:
import requests


API_KEY = "" 
API_ENDPOINT = ""

def rewrite_description(description):
    """
    Use the API to rewrite a description in a clear, professional, plain text format.
    
    Args:
        description (str): The original description text.
        
    Returns:
        str: The rewritten description from the model in plain text, enclosed in quotes.
    """
   
    prompt = f"Rewrite the following description in clear, professional, plain text. Avoid any special characters, lists, or formatting and don't need to mentioned that this is belong to image or scense and just focus on the content. The output should be a single plain paragraph:\n\n{description}"
    
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }

    
    data = {
        "model": "gpt-4o-mini",  
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
    }

    try:
       
        response = requests.post(API_ENDPOINT, json=data, headers=headers)
        response.raise_for_status()  
        response_data = response.json()

        rewritten_description = response_data.get("choices")[0].get("message").get("content").strip()
        return f'"{rewritten_description}"'

    except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
        print(f"Response content: {response.content.decode('utf-8')}")
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

def process_file(file_path):
    """
    Read descriptions from a file, rewrite them using the API, and print the results.
    
    Args:
        file_path (str): Path to the text file containing descriptions.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            lines = file.readlines()
        
        image_name = ""
        description_lines = []

    
        for line in lines:
            line = line.strip()  

            if not line:  
                if image_name and description_lines:  
                    description = " ".join(description_lines).strip()
                    print(f"Image: {image_name}")
                    rewritten_desc = rewrite_description(description)
                    if rewritten_desc:
                        print(f"Rewritten Description: {rewritten_desc}")
                    print("-" * 50)
                
                image_name = ""
                description_lines = []
            elif line.startswith("Image:"): 
                image_name = line[len("Image:"):].strip().split("/")[-1] 
            elif line.startswith("Description:"):  
                description_lines.append(line[len("Description:"):].strip())
            else:
                
                description_lines.append(line)

        if image_name and description_lines:
            description = " ".join(description_lines).strip()
            print(f"Image: {image_name}")
            rewritten_desc = rewrite_description(description)
            if rewritten_desc:
                print(f"Rewritten Description: {rewritten_desc}")
            print("-" * 50)

    except FileNotFoundError:
        print(f"Error: The file at '{file_path}' was not found.")
    except Exception as e:
        print(f"An error occurred while reading the file: {e}")


process_file(r"10\New Text Document10.txt")  

Image: images (4).jpeg
Rewritten Description: "A person's hand is placing a wooden block with a question mark on top of a stack of similar blocks that are arranged vertically on a wooden surface. This action suggests a decision-making or problem-solving scenario, as the question marks on the blocks signify uncertainty or a need for additional information or clarification. The setting appears to be indoors, likely on a table or desk."
--------------------------------------------------
Image: download (31).jpeg
Rewritten Description: "The chalkboard displays a range of business and marketing concepts, featuring various icons, diagrams, and text that illustrate different strategies and ideas. In the foreground, elements such as a light bulb and a cup suggest a relaxed setting, possibly a coffee shop or casual workspace. The concepts on the chalkboard include technology, indicated by a computer icon, data, represented by a graph or chart, analytics, symbolized by a pie chart, and social me

In [32]:
# Open and read the file
with open('10/rewritten description10.txt', 'r', encoding='utf-8') as file:
    content = file.read()

# Split the content into blocks by the separator
blocks = content.split('--------------------------------------------------')

# Extract the text after "Rewritten Description: " from each block
rewritten_descriptions = []
for block in blocks:
    if "Rewritten Description:" in block:
        start_index = block.find("Rewritten Description:") + len("Rewritten Description:")
        rewritten_text = block[start_index:].strip()
        rewritten_descriptions.append(rewritten_text)

# Join the descriptions with commas and print
output = ', '.join(rewritten_descriptions)
for item in rewritten_descriptions:
    print(f"{item},")


"A person's hand is placing a wooden block with a question mark on top of a stack of similar blocks that are arranged vertically on a wooden surface. This action suggests a decision-making or problem-solving scenario, as the question marks on the blocks signify uncertainty or a need for additional information or clarification. The setting appears to be indoors, likely on a table or desk.",
"The chalkboard displays a range of business and marketing concepts, featuring various icons, diagrams, and text that illustrate different strategies and ideas. In the foreground, elements such as a light bulb and a cup suggest a relaxed setting, possibly a coffee shop or casual workspace. The concepts on the chalkboard include technology, indicated by a computer icon, data, represented by a graph or chart, analytics, symbolized by a pie chart, and social media, among others.",
"A man is running on a curved red path that leads to a flag at the end. The path contains several small circular obstacles, 

In [ ]:
# !pip install sentence_transformers

In [33]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
import numpy as np

# Example inputs
responses = rewritten_descriptions

# Step 1: Load SentenceTransformer model
model = SentenceTransformer('all-MiniLM-L6-v2')  # You can use any pre-trained model

# Step 2: Compute embeddings
response_embeddings = model.encode(responses)

# Step 3: Select diverse responses using clustering
num_responses_to_select = 50
if len(responses) > num_responses_to_select:
    # Use KMeans to find diverse clusters
    kmeans = KMeans(n_clusters=num_responses_to_select, random_state=42)
    kmeans.fit(response_embeddings)
    
    # Get one response per cluster (closest to cluster center)
    diverse_responses = []
    for cluster_idx in range(num_responses_to_select):
        cluster_indices = np.where(kmeans.labels_ == cluster_idx)[0]
        cluster_center = kmeans.cluster_centers_[cluster_idx]
        closest_idx = min(cluster_indices, key=lambda idx: np.linalg.norm(response_embeddings[idx] - cluster_center))
        diverse_responses.append(responses[closest_idx])
else:
    # If fewer than 50 responses, return all
    diverse_responses = responses

# Step 4: Output the results
print("Selected 50 diverse responses:")
for i, response in enumerate(diverse_responses, 1):
    print(f"{response},")

Selected 50 diverse responses:
"A person's hand is placing a wooden block with a question mark on top of a stack of similar blocks that are arranged vertically on a wooden surface. This action suggests a decision-making or problem-solving scenario, as the question marks on the blocks signify uncertainty or a need for additional information or clarification. The setting appears to be indoors, likely on a table or desk.",
"The chalkboard displays a range of business and marketing concepts, featuring various icons, diagrams, and text that illustrate different strategies and ideas. In the foreground, elements such as a light bulb and a cup suggest a relaxed setting, possibly a coffee shop or casual workspace. The concepts on the chalkboard include technology, indicated by a computer icon, data, represented by a graph or chart, analytics, symbolized by a pie chart, and social media, among others.",
"A man is running on a curved red path that leads to a flag at the end. The path contains sev